In [4]:
# import tensorflow as tf

# print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

In [5]:
# !nvidia-smi

# Tahap 2 — Download Poster Tambahan (MovieGenre.csv)

Notebook ini mengunduh poster film tambahan dari URL yang sudah tersedia di `MovieGenre.csv`
(kolom `Poster`), memprioritaskan genre minoritas (Animation, Action, Horror, Adventure)
supaya dataset akhir tidak setimpang sample awal (997 gambar, Drama 601 vs Animation 28).

**Sebelum run:** pastikan `MovieGenre.csv` sudah ada di direktori kerja Colab
(via upload manual, atau mount Google Drive).


**Catatan path:** notebook ini diasumsikan dijalankan dari folder `notebooks/` pada struktur project lokal (bukan root Colab). `MovieGenre.csv` dibaca dari `../data/raw/`, poster disimpan ke `../data/posters/`, dan log unduhan ke `../outputs/logs/`. Kalau dijalankan langsung di Colab (bukan lewat mount project ini), sesuaikan ulang variabel `CSV_PATH` dan `OUT_DIR` ke path yang sesuai di sana.

In [1]:
import os
import time
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

## Konfigurasi

In [2]:
import sys

# Deteksi otomatis: Colab atau lokal
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files

    # Upload MovieGenre.csv kalau belum ada
    if not os.path.exists("MovieGenre.csv"):
        print("Upload file MovieGenre.csv dari komputer lokal:")
        uploaded = files.upload()

    CSV_PATH = "MovieGenre.csv"
    OUT_DIR = "posters"
else:
    CSV_PATH = "../data/raw/MovieGenre.csv"
    OUT_DIR = "../data/posters"

TARGET_GENRES = ["Action", "Adventure", "Animation", "Comedy",
                  "Drama", "Horror", "Romance"]
MAX_PER_GENRE = 400                  # atur sesuai kebutuhan/waktu unduh
TIMEOUT = 8
MAX_WORKERS = 16

os.makedirs(OUT_DIR, exist_ok=True)
print(f"CSV_PATH = {CSV_PATH}")
print(f"OUT_DIR  = {OUT_DIR}")

CSV_PATH = ../data/raw/MovieGenre.csv
OUT_DIR  = ../data/posters


## Load &amp; Filter Dataset

Ambil baris dengan genre &amp; link poster valid, lalu simpan hanya genre yang termasuk 7 target.

In [3]:
df = pd.read_csv(CSV_PATH, encoding="latin1")
df = df.dropna(subset=["Genre", "Poster"])
df["genre_list"] = df["Genre"].astype(str).str.split("|")
df["target_genres"] = df["genre_list"].apply(
    lambda gl: [g for g in gl if g in TARGET_GENRES]
)
df = df[df["target_genres"].apply(len) > 0].copy()

print("Total baris dengan minimal 1 genre target:", len(df))

Total baris dengan minimal 1 genre target: 34527


## Prioritaskan Genre Minoritas

Kalau diunduh urut apa adanya, kuota `MAX_PER_GENRE` akan cepat habis oleh Drama/Comedy
(paling banyak jumlahnya), sementara Animation/Action/Horror/Adventure (paling sedikit)
tidak kebagian jatah. Jadi baris dengan genre langka diproses lebih dulu.

In [3]:
genre_counts = {g: 0 for g in TARGET_GENRES}
minority_order = ["Animation", "Action", "Horror", "Adventure",
                   "Romance", "Comedy", "Drama"]

def genre_priority(genres):
    ranks = [minority_order.index(g) for g in genres if g in minority_order]
    return min(ranks) if ranks else 99

df["priority"] = df["target_genres"].apply(genre_priority)
df = df.sort_values("priority")

selected_rows = []
for _, row in df.iterrows():
    gl = row["target_genres"]
    if any(genre_counts[g] < MAX_PER_GENRE for g in gl):
        selected_rows.append(row)
        for g in gl:
            genre_counts[g] += 1

print("Rencana unduh:", len(selected_rows), "poster")
print("Estimasi per genre (bisa overlap krn multi-label):", genre_counts)

NameError: name 'TARGET_GENRES' is not defined

## Fungsi Download Satu Poster

In [10]:
def download_one(row):
    imdb_id = str(row["imdbId"])
    url = row["Poster"]
    out_path = os.path.join(OUT_DIR, f"{imdb_id}.jpg")
    if os.path.exists(out_path):
        return imdb_id, "skip_exists"
    try:
        r = requests.get(url, timeout=TIMEOUT)
        if r.status_code == 200 and r.content:
            with open(out_path, "wb") as f:
                f.write(r.content)
            return imdb_id, "ok"
        else:
            return imdb_id, f"http_{r.status_code}"
    except Exception as e:
        err_name = type(e).__name__
        return imdb_id, "error_" + err_name

## Jalankan Download Paralel

Memakai 16 thread sekaligus supaya tidak menunggu satu-satu.

In [11]:
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(download_one, row): row for _, row in
               pd.DataFrame(selected_rows).iterrows()}
    for i, fut in enumerate(as_completed(futures), 1):
        imdb_id, status = fut.result()
        results.append((imdb_id, status))
        if i % 200 == 0:
            print(f"progress: {i}/{len(selected_rows)}")

progress: 200/2071
progress: 400/2071
progress: 600/2071
progress: 800/2071
progress: 1000/2071
progress: 1200/2071
progress: 1400/2071
progress: 1600/2071
progress: 1800/2071
progress: 2000/2071


## Ringkasan &amp; Log Hasil

Catat status tiap unduhan (berhasil / link mati / error) supaya bisa diaudit.

In [12]:
res_df = pd.DataFrame(results, columns=["imdbId", "status"])
print(res_df["status"].value_counts())

LOG_DIR = "logs" if IN_COLAB else "../outputs/logs"
os.makedirs(LOG_DIR, exist_ok=True)
log_path = os.path.join(LOG_DIR, "download_log_1b.csv")
res_df.to_csv(log_path, index=False)

ok_ids = set(res_df.loc[res_df["status"] == "ok", "imdbId"])
print("Berhasil diunduh:", len(ok_ids), "poster baru")
print(f"File log tersimpan di {log_path} untuk audit link mati.")

status
ok                   1542
http_404              503
skip_exists            23
error_ReadTimeout       2
http_403                1
Name: count, dtype: int64
Berhasil diunduh: 1539 poster baru
File log tersimpan di ../outputs/logs\download_log_1b.csv untuk audit link mati.


## Verifikasi Distribusi Genre Riil

Target `MAX_PER_GENRE` dihitung dari ketersediaan baris di CSV **sebelum** download,
bukan dari hasil download yang sukses. Karena ada link mati (`http_404`, dll), jumlah
riil per genre setelah download bisa lebih kecil dari target — terutama untuk genre
minoritas (Animation, Action). Cell ini juga mengecek apakah ada `imdbId` duplikat
di dataset final.

In [13]:
import os
from collections import Counter

downloaded_ids = set(f.split(".")[0] for f in os.listdir(OUT_DIR))
final_df = df[df["imdbId"].astype(str).isin(downloaded_ids)]

# cek duplikat imdbId
dup_count = final_df["imdbId"].duplicated().sum()
print("Baris dengan imdbId duplikat:", dup_count)

# hitung distribusi genre riil (setelah drop duplikat)
final_df = final_df.drop_duplicates(subset="imdbId")
c = Counter()
for gl in final_df["target_genres"]:
    for g in gl:
        c[g] += 1
print("Distribusi genre riil setelah download:")
for g in TARGET_GENRES:
    print(f"  {g}: {c.get(g, 0)}")

Baris dengan imdbId duplikat: 26
Distribusi genre riil setelah download:
  Action: 862
  Adventure: 530
  Animation: 845
  Comedy: 441
  Drama: 398
  Horror: 378
  Romance: 276


# Tahap 3 — Gabungkan Semua Poster & Bersihkan Duplikat (Deduplikasi 3 Lapis)

Tahap ini memvalidasi integritas file poster, mendeteksi duplikasi melalui 3 lapis:
1. **Lapis 1:** `imdbId` yang sama pada dataset.
2. **Lapis 2:** Byte Hash (MD5) untuk mendeteksi file poster identik secara biner.
3. **Lapis 3:** Visual Perceptual Hash (dHash) untuk mendeteksi poster yang secara visual sama persis.

File yang duplikat atau corrupt dipindahkan ke folder `quarantine/` (non-destruktif), dan data bersih disimpan ke `data/processed/clean_posters_metadata.csv`.

In [4]:
import os
import sys
import pandas as pd

# Konfigurasi Path Tahap 3
LOG_DIR = "logs" if IN_COLAB else "../outputs/logs"
os.makedirs(LOG_DIR, exist_ok=True)
QUARANTINE_DIR = "quarantine" if IN_COLAB else "../data/quarantine"
PROCESSED_DIR = "processed" if IN_COLAB else "../data/processed"
CLEAN_META_PATH = os.path.join(PROCESSED_DIR, "clean_posters_metadata.csv")
AUDIT_LOG_PATH = os.path.join(LOG_DIR, "stage3_dedup_audit.csv")

os.makedirs(QUARANTINE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"OUT_DIR         : {OUT_DIR}")
print(f"QUARANTINE_DIR  : {QUARANTINE_DIR}")
print(f"CLEAN_META_PATH : {CLEAN_META_PATH}")
print(f"AUDIT_LOG_PATH  : {AUDIT_LOG_PATH}")

OUT_DIR         : ../data/posters
QUARANTINE_DIR  : ../data/quarantine
CLEAN_META_PATH : ../data/processed\clean_posters_metadata.csv
AUDIT_LOG_PATH  : ../outputs/logs\stage3_dedup_audit.csv


In [5]:
# Import fungsi deduplikasi dari modul reusable src/utils/dedup.py
for _p in ("../src", "src"):
    if os.path.exists(os.path.join(_p, "utils", "dedup.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
from utils.dedup import run_stage3_pipeline
print("Modul dedup dimuat dari:", os.path.abspath(_p))

Modul dedup dimuat dari: c:\Kodingan\skripsi-genre-klasifikasi\src


In [6]:
# Eksekusi Deduplikasi 3 Lapis & Pembersihan (via run_stage3_pipeline)
df_valid, df_audit = run_stage3_pipeline(
    posters_dir=OUT_DIR,
    df_metadata=df,
    quarantine_dir=QUARANTINE_DIR,
    output_metadata_path=CLEAN_META_PATH,
    log_audit_path=AUDIT_LOG_PATH,
)

Total file gambar ditemukan: 1535

=== RINGKASAN TAHAP 3 ===
Total file diproses      : 1535
Total poster valid       : 1535
Total dikarantina        : 0
Log audit disimpan di    : ../outputs/logs\stage3_dedup_audit.csv
Metadata bersih di       : ../data/processed\clean_posters_metadata.csv


# Tahap 4 — Cek Jumlah Akhir per Genre

Validasi distribusi genre final **setelah gabungan poster lama + baru melewati deduplikasi 3 lapis (Tahap 3)**.
Berbeda dengan verifikasi di akhir Tahap 2 (cek hasil download), tahap ini memakai
`clean_posters_metadata.csv` — dataset bersih yang akan jadi input Tahap 5+.

Keputusan setelah tahap ini:
- Distribusi cukup seimbang → lanjut Tahap 5 (bersihkan data).
- Masih timpang ekstrem → unduh poster tambahan (kembali ke Tahap 2) atau terima ketimpangan
  dan catat sebagai limitasi di skripsi.

In [7]:
# Tahap 4: Distribusi genre final dari metadata bersih hasil Tahap 3
from collections import Counter

df_clean = pd.read_csv(CLEAN_META_PATH)

print(f"Total poster bersih     : {len(df_clean)}")
print(f"Baris duplikat imdbId   : {df_clean['imdbId'].duplicated().sum()}")
print()

# Distribusi genre (kolom Genre dari metadata gabungan)
c = Counter()
for genres in df_clean["Genre"].dropna():
    for g in str(genres).split(","):
        g = g.strip()
        if g in TARGET_GENRES:
            c[g] += 1

print("Distribusi genre final (7 genre target):")
for g in TARGET_GENRES:
    n = c.get(g, 0)
    bar = "#" * max(1, n // 25)
    print(f"  {g:<12} {n:>5}  {bar}")

total = sum(c.values())
print(f"\nTotal label genre (multi-label, satu poster bisa punya >1): {total}")
mx, mn = max(c.values()), min(c.values())
print(f"Rasio genre terbesar/terkecil: {mx/mn:.1f}:1  (max={mx}, min={mn})")
if mx/mn > 3:
    print("WARNING: distribusi masih timpang (>3:1). Pertimbangkan tambah data genre minoritas atau catat sebagai limitasi.")
else:
    print("Distribusi cukup seimbang (<3:1). Lanjut Tahap 5.")

Total poster bersih     : 1535
Baris duplikat imdbId   : 0

Distribusi genre final (7 genre target):
  Action           0  #
  Adventure        0  #
  Animation        2  #
  Comedy           0  #
  Drama            0  #
  Horror           0  #
  Romance          0  #

Total label genre (multi-label, satu poster bisa punya >1): 2
Rasio genre terbesar/terkecil: 1.0:1  (max=2, min=2)
Distribusi cukup seimbang (<3:1). Lanjut Tahap 5.
